In [ ]:
!pip install meteostat

In [ ]:
!pip -q install "pandas==2.2.2" "meteostat==1.6.8"

In [ ]:
from datetime import datetime
from meteostat import Stations, Daily
import pandas as pd

# Boston latitude and longitude range
LAT_MIN, LAT_MAX = 42.15, 42.55
LON_MIN, LON_MAX = -71.35, -70.85

start = datetime(2018, 1, 1)
end   = datetime(2026, 2, 7)


stations = (
    Stations()
    .bounds((LAT_MAX, LON_MIN), (LAT_MIN, LON_MAX))
    .fetch()
)


daily = Daily(stations, start, end).fetch()

# multi-index for daily data
# Avg precipitation across stations for each day
daily_prcp = (
    daily['prcp']
    .groupby('time')
    .mean()
    .to_frame('prcp')
)
daily_prcp.index.name = 'date'

daily_prcp.to_csv("/content/boston_daily_precip_2018_2026_bbox.csv")
from google.colab import files
files.download("/content/boston_daily_precip_2018_2026_bbox.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from datetime import date
import meteostat as ms

# (Greater) Boston latitude and longitude range
LAT_MIN, LAT_MAX = 42.15, 42.55
LON_MIN, LON_MAX = -71.35, -70.85

start = date(2018, 1, 1)
end = date(2026, 2, 7)

# Filtering nearby stations
point = ms.Point(42.3601, -71.0589)
stations = ms.stations.nearby(point, limit=50)

stations = stations[
    stations['latitude'].between(LAT_MIN, LAT_MAX) &
    stations['longitude'].between(LON_MIN, LON_MAX)
]


# rhum (relative humidity) is included in this
ts = ms.hourly(stations, start, end)
hourly = ts.fetch()

daily_region = (
    hourly['rhum']
      .groupby('station')
      .resample('D', level='time')
      .mean()
      .groupby('time')
      .mean()
      .rename('rhum')
)

pd.set_option("display.max_rows", None)
print(daily_region)

sheet_df = daily_region.rename("rhum").to_frame()
sheet_df.index.name = "date"


sheet_df = sheet_df.loc["2018":"2026"]

# downloading the file
path = "/content/boston_rhum_2018_2026.csv"
sheet_df.to_csv(path)

from google.colab import files
files.download(path)

time
2018-01-01    48.807367
2018-01-02    57.347222
2018-01-03    54.833333
2018-01-04         77.0
2018-01-05       55.625
2018-01-06    44.208333
2018-01-07    41.166667
2018-01-08    46.402778
2018-01-09    64.444444
2018-01-10    60.138889
2018-01-11    80.361111
2018-01-12         94.5
2018-01-13    81.486111
2018-01-14    49.319444
2018-01-15    58.541667
2018-01-16    61.027778
2018-01-17    86.916667
2018-01-18    74.361111
2018-01-19    67.027778
2018-01-20    54.472222
2018-01-21    62.916667
2018-01-22    89.652778
2018-01-23    96.755435
2018-01-24    70.055556
2018-01-25       44.875
2018-01-26    36.208333
2018-01-27    59.430556
2018-01-28    72.583333
2018-01-29    72.638889
2018-01-30    65.083333
2018-01-31    44.208333
2018-02-01    58.972222
2018-02-02    71.291667
2018-02-03    47.430556
2018-02-04    70.791667
2018-02-05    77.222222
2018-02-06    53.055556
2018-02-07    82.916667
2018-02-08       71.875
2018-02-09    62.819444
2018-02-10    81.111111
2018-02-11 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
sheet_df = daily_region.rename("rhum").to_frame()
sheet_df.index.name = "date"


sheet_df = sheet_df.loc["2018":"2026"]

# download
path = "/content/boston_rhum_2018_2026.csv"
sheet_df.to_csv(path)

from google.colab import files
files.download(path)

TypeError: 'str' object is not callable

In [ ]:
import meteostat as ms
ms.config.block_large_requests = False

ModuleNotFoundError: No module named 'meteostat'

In [ ]:
from datetime import date
import pandas as pd
import meteostat as ms
from google.colab import files

LAT_MIN, LAT_MAX = 42.15, 42.55
LON_MIN, LON_MAX = -71.35, -70.85

start = date(2023, 3, 2)
end   = date(2026, 2, 7)

point = ms.Point(42.3601, -71.0589)
stations = ms.stations.nearby(point, limit=50)

stations = stations[
    stations['latitude'].between(LAT_MIN, LAT_MAX) &
    stations['longitude'].between(LON_MIN, LON_MAX)
]

# (temp in °C, wspd in km/h)
ts = ms.hourly(stations, start, end)
hourly = ts.fetch()

# Daily mean temp and wind per station, then regional mean
daily_station = (
    hourly[['temp', 'wspd']]
      .groupby('station')
      .resample('D', level='time')
      .mean()
)

daily_region = (
    daily_station
      .groupby('time')
      .mean()
      .rename(columns={'temp': 'temp_c', 'wspd': 'wind_kph'})
      .sort_index()
)


daily_region = daily_region.loc["2023-03-02":"2026-02-07"]

# download
path = "/content/boston_temp_wind_2023-03-02_to_2026-02-07.csv"
daily_region.to_csv(path)
files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from datetime import date
import numpy as np
import pandas as pd
import meteostat as ms

LAT_MIN, LAT_MAX = 42.15, 42.55
LON_MIN, LON_MAX = -71.35, -70.85

point = ms.Point(42.3601, -71.0589)
stations = ms.stations.nearby(point, limit=50)
stations = stations[
    stations["latitude"].between(LAT_MIN, LAT_MAX) &
    stations["longitude"].between(LON_MIN, LON_MAX)
]

# Split into <=3-year chunks
chunks = [
    (date(2020, 1, 1), date(2022, 12, 31)),
    (date(2023, 1, 1), date(2025, 12, 31)),
    (date(2026, 1, 1), date(2026, 2, 7)),
]

parts = []
for s, e in chunks:
    h = ms.hourly(stations, s, e).fetch()
    if not h.empty:
        parts.append(h)

hourly = pd.concat(parts).sort_index()

# Wind direction only
hourly = hourly[["wdir"]].dropna().copy()

# Circular mean
hourly["rad"] = np.deg2rad(hourly["wdir"])
hourly["sin"] = np.sin(hourly["rad"])
hourly["cos"] = np.cos(hourly["rad"])

# Daily circular mean by station
station_day = (
    hourly.groupby("station")
    .resample("D", level="time")[["sin", "cos"]]
    .mean()
)

# Daily circular mean across stations
region_day = station_day.groupby("time")[["sin", "cos"]].mean()

region_day["wind_dir_deg"] = (
    np.rad2deg(np.arctan2(region_day["sin"], region_day["cos"])) + 360
) % 360

daily_region = region_day[["wind_dir_deg"]].sort_index()
daily_region = daily_region.loc["2020-01-01":"2026-02-07"]

# Optional: force full daily index
full_idx = pd.date_range("2020-01-01", "2026-02-07", freq="D")
daily_region = daily_region.reindex(full_idx)
daily_region.index.name = "time"

out_path = "/content/boston_wind_direction_2020-01-01_to_2026-02-07.csv"
daily_region.to_csv(out_path)

print("Saved to:", out_path)
print("Exists:", pd.io.common.file_exists(out_path))
print("Rows:", len(daily_region))

files.download(out_path)


Saved to: /content/boston_wind_direction_2020-01-01_to_2026-02-07.csv
Exists: True
Rows: 2230


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>